In [16]:
import pandas as pd
import json

# Load the CSV file
df = pd.read_csv("../project_data.csv")

# Display the first few rows
print(df.head())

   project_id project_type                                    title  \
0           1      Theater            Sharable bifurcated algorithm   
1           2      Theater  Face-to-face systemic Graphic Interface   
2           3           TV         Organized bottom-line encryption   
3           4      Theater              Diverse analyzing migration   
4           5         Film   Multi-channeled demand-driven software   

                                         description  start_date    end_date  \
0  Development say quality throughout beautiful. ...  2024-06-23  2024-10-06   
1  Start source husband at tree. Then fire pretty...  2025-02-03  2025-05-31   
2  Full open discover detail. Remain arrive attac...  2025-04-10  2025-08-27   
3  Attention attack technology identify. East org...  2023-07-14  2023-12-07   
4  Government nice themselves wind. Understand do...  2023-11-14  2024-04-01   

                                            synopsis  plagiarism_similarity  \
0  Stop peace

In [4]:
print(df['review_status'].value_counts())

review_status
Approved        51
Needs Review    25
Rejected        24
Name: count, dtype: int64


In [5]:
# Check missing values in text columns
print(df[['title', 'description', 'synopsis']].isnull().sum())

# Check average text length
print("Average title length:", df['title'].str.len().mean())
print("Average description length:", df['description'].str.len().mean())

title          0
description    0
synopsis       0
dtype: int64
Average title length: 34.56
Average description length: 145.13


In [6]:
# Convert string to JSON
df['production_dates'] = df['production_dates'].apply(json.loads)

# Extract duration in days
def get_duration_days(production_dates):
    for entry in production_dates:
        if 'duration' in entry:
            months = int(entry['duration'].split()[0])
            return months * 30  # Convert months to days
    return 0

df['duration_days'] = df['production_dates'].apply(get_duration_days)

JSONDecodeError: Expecting property name enclosed in double quotes: line 1 column 3 (char 2)

In [7]:
# Display unique values in production_dates
print(df['production_dates'].unique())

["[{'dateType': 'Rehearsal Dates', 'dateRange': ['2024-07-26', '2024-08-05']}]"
 "[{'dateType': 'Shooting Dates', 'dateRange': ['2025-03-26', '2025-04-04']}]"
 "[{'dateType': 'Shooting Dates', 'dateRange': ['2025-07-31', '2025-08-03']}, {'dateType': 'Rehearsal Dates', 'dateRange': ['2025-07-24', '2025-07-29']}, {'dateType': 'Shooting Dates', 'dateRange': ['2025-04-21', '2025-04-25']}]"
 "[{'dateType': 'Rehearsal Dates', 'dateRange': ['2023-10-29', '2023-11-04']}, {'dateType': 'Rehearsal Dates', 'dateRange': ['2023-08-11', '2023-08-12']}]"
 "[{'dateType': 'Shooting Dates', 'dateRange': ['2024-02-26', '2024-03-06']}, {'dateType': 'Shooting Dates', 'dateRange': ['2024-03-26', '2024-04-02']}, {'dateType': 'Audition Dates', 'dateRange': ['2024-03-23', '2024-04-02']}]"
 "[{'dateType': 'Audition Dates', 'dateRange': ['2023-07-29', '2023-08-07']}, {'dateType': 'Audition Dates', 'dateRange': ['2023-08-15', '2023-08-17']}]"
 "[{'dateType': 'Shooting Dates', 'dateRange': ['2023-06-19', '2023-06-2

In [8]:
import json

def safe_json_loads(json_str):
    try:
        # Attempt to load JSON string
        return json.loads(json_str) if isinstance(json_str, str) else {}
    except (ValueError, TypeError):
        # Return an empty dictionary for invalid JSON
        return {}

# Apply the function to the production_dates column
df['production_dates'] = df['production_dates'].apply(safe_json_loads)

In [9]:
def get_duration_days(production_dates):
    if not isinstance(production_dates, dict):  # Ensure it's a dictionary
        return 0
    if 'duration' in production_dates:
        try:
            months = int(production_dates['duration'].split()[0])
            return months * 30  # Convert months to days
        except (ValueError, AttributeError):
            return 0  # Handle invalid duration formats
    return 0  # Default if 'duration' is missing

In [10]:
def get_duration_days(production_dates):
    if not isinstance(production_dates, dict):  # Ensure it's a dictionary
        return 0
    if 'duration' in production_dates:
        try:
            months = int(production_dates['duration'].split()[0])
            return months * 30  # Convert months to days
        except (ValueError, AttributeError):
            return 0  # Handle invalid duration formats
    return 0  # Default if 'duration' is missing

In [11]:
df['duration_days'] = df['production_dates'].apply(get_duration_days)

In [12]:
# Display the first few rows
print(df[['production_dates', 'duration_days']].head())

# Check for null or zero values in duration_days
print(df[df['duration_days'] == 0]['production_dates'].head())

  production_dates  duration_days
0               {}              0
1               {}              0
2               {}              0
3               {}              0
4               {}              0
0    {}
1    {}
2    {}
3    {}
4    {}
Name: production_dates, dtype: object


In [13]:
print(df[pd.isnull(df['production_dates']) | (df['production_dates'].apply(lambda x: isinstance(x, str)) & df['production_dates'].str.contains('error'))])

Empty DataFrame
Columns: [project_id, project_type, title, description, start_date, end_date, synopsis, plagiarism_similarity, production_dates, production_locations, crew_roles, casting_roles, review_status, duration_days]
Index: []


In [6]:
# Check if critical roles exist (e.g., "Director")
df['has_director'] = df['crew_roles'].apply(
    lambda roles: 1 if 'Director' in roles else 0
)

In [7]:
# Count locations
df['location_count'] = df['production_locations'].apply(lambda x: len(json.loads(x)))

# Create dummy variables for key locations
df['has_colombo'] = df['production_locations'].apply(
    lambda locs: 1 if 'Colombo' in json.loads(locs) else 0
)

JSONDecodeError: Expecting property name enclosed in double quotes: line 1 column 3 (char 2)

In [8]:
# Check missing values
print(df.isnull().sum())

# Fill missing dates with default values (if any)
df['start_date'].fillna('2020-01-01', inplace=True)

# Fill text features with empty strings
df['synopsis'].fillna('', inplace=True)

# Replace missing categorical values with "Unknown"
df['project_type'].fillna('Unknown', inplace=True)

project_id               0
project_type             0
title                    0
description              0
start_date               0
end_date                 0
synopsis                 0
plagiarism_similarity    0
production_dates         0
production_locations     0
crew_roles               0
casting_roles            0
review_status            0
has_director             0
dtype: int64


C:\Users\DELL\AppData\Local\Temp\ipykernel_1844\1586536532.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['start_date'].fillna('2020-01-01', inplace=True)
C:\Users\DELL\AppData\Local\Temp\ipykernel_1844\1586536532.py:8: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For exam

In [9]:
from datetime import datetime

# Convert string dates to datetime
df['start_date'] = pd.to_datetime(df['start_date'])
df['end_date'] = pd.to_datetime(df['end_date'])

# Calculate project duration
df['project_duration'] = (df['end_date'] - df['start_date']).dt.days

# Example: Parse duration strings (if needed)
# df['duration_days'] = df['production_dates'].apply(extract_duration_days)

In [10]:
df.to_csv("project_data_preprocessed.csv", index=False)